# EDB Data Scientist Assessment — Part A
## ML Pipeline: Company Churn Prediction

**Time allowed:** 30 minutes  
**Total marks:** 40 pts  

### Context
EDB tracks whether companies stay engaged with its programmes ('churned' = 1 means the company disengaged). Your task is to build a clean, well-evaluated classification pipeline on the provided dataset `companies.csv`.

### Instructions
- Complete each section below in order.
- Leave comments explaining your decisions — especially any choices you made about how to handle data issues.
- Do **not** modify the evaluation cell at the very end.
- Internet access is allowed. You may use any scikit-learn-compatible library.

---

In [ ]:
# Run this cell first — installs / imports all dependencies
# !pip install pandas numpy scikit-learn xgboost matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import cross_val_score, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

print('All imports successful')

---
## A1 · Data Cleaning  *(10 pts)*

Load `companies.csv` and investigate its quality. You should find and address **at least 3 distinct data quality issues**.

For each issue:
1. Show evidence it exists (print, describe, or visualise)
2. Apply your fix
3. Add a comment explaining *why* you chose that approach

Do **not** drop the `churned` column — that is your target.

In [ ]:
# Load data
df = pd.read_csv('companies.csv')
print(df.shape)
df.head()

In [ ]:
# A1 — Your data cleaning code here
# Hint: check dtypes, nulls, duplicates, value distributions, and string inconsistencies

# ----- Issue 1: ------------------------------------------------
# Evidence:

# Fix:

# ----- Issue 2: ------------------------------------------------
# Evidence:

# Fix:

# ----- Issue 3: ------------------------------------------------
# Evidence:

# Fix:

# (Add more issues if you find them)

df_clean = df.copy()  # replace with your cleaned dataframe
print(f'Clean shape: {df_clean.shape}')

---
## A2 · Feature Engineering  *(10 pts)*

- Create **at least 2 meaningful derived features** and explain what they represent.
- Build a scikit-learn `ColumnTransformer` / `Pipeline` that handles:
  - Remaining nulls (imputation)
  - Categorical encoding
  - Numeric scaling
- Avoid data leakage — transformations must be fit only on training data.

In [ ]:
# A2 — Feature engineering

# --- Derived feature 1: ---
# Rationale:

# --- Derived feature 2: ---
# Rationale:

# Define feature columns
NUMERIC_FEATURES = []  # fill in
CATEGORICAL_FEATURES = []  # fill in
TARGET = 'churned'

X = df_clean[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df_clean[TARGET]

print(f'X shape: {X.shape}, class balance: {y.value_counts(normalize=True).to_dict()}')

In [ ]:
# Build preprocessing pipeline
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_transformer = Pipeline(steps=[
    # TODO: add imputer and scaler
])

categorical_transformer = Pipeline(steps=[
    # TODO: add imputer and encoder
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, NUMERIC_FEATURES),
    ('cat', categorical_transformer, CATEGORICAL_FEATURES),
])

print('Preprocessor defined')

---
## A3 · Model Training & Evaluation  *(15 pts)*

- Train **at least 2 classifiers** using your pipeline.
- Evaluate using **5-fold stratified cross-validation**.
- Report: **Precision, Recall, F1, ROC-AUC** for both models.
- Produce at least one visualisation: confusion matrix **or** ROC curve.
- Choose a final model and briefly justify the choice.

In [ ]:
# A3 — Model training
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Model 1
model_1 = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Model 2
model_2 = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'))
])

# TODO: run cross_validate with scoring=['precision','recall','f1','roc_auc']
# and print a summary comparison table

In [ ]:
# Visualisation — confusion matrix or ROC curve for your chosen model
# TODO

---
## A4 · Insight Commentary  *(5 pts)*

In 2–4 sentences answer:
1. Based on your model or EDA, what are the **top drivers of churn**?
2. What would you try **with more time** (data, features, modelling, deployment)?

*Write your answer in this markdown cell.*

---

**Your answer here.**

---

In [ ]:
# ============================================================
# DO NOT MODIFY — Evaluator check cell
# ============================================================
checks = {}

# Check 1: df_clean exists and has fewer rows than raw (duplicates removed)
try:
    checks['duplicates_removed'] = len(df_clean) < len(df)
except:
    checks['duplicates_removed'] = False

# Check 2: Target column present
checks['target_present'] = 'churned' in df_clean.columns

# Check 3: At least 2 numeric features defined
checks['features_defined'] = len(NUMERIC_FEATURES) >= 2 and len(CATEGORICAL_FEATURES) >= 1

# Check 4: Preprocessor has both transformers
checks['preprocessor_has_num_cat'] = len(preprocessor.transformers) == 2

for k, v in checks.items():
    status = '✓' if v else '✗'
    print(f'{status}  {k}')